# imports

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sqlalchemy import create_engine

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# conexión a PostgreSQL

In [2]:
DB_USER = "penguins"
DB_PASSWORD = "penguins123"
DB_HOST = "penguins_db"
DB_PORT = "5432"
DB_NAME = "penguins_db"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Conexión a PostgreSQL OK")

Conexión a PostgreSQL OK


# conexión a MLflow

In [3]:
mlflow.set_tracking_uri("http://mlflow:5000")
mlflow.set_experiment("penguins_classification")

print("MLflow conectado")

MLflow conectado


# cargar el CSV desde la carpeta notebooks

In [4]:
df_csv = pd.read_csv("/app/notebooks/penguins_v1.csv")
df_csv.head()

,id,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,1,1,1,39.1,18.7,181,3750,1,2007
1,2,1,1,39.5,17.4,186,3800,0,2007
2,3,1,1,40.3,18.0,195,3250,0,2007
3,5,1,1,36.7,19.3,193,3450,0,2007
4,6,1,1,39.3,20.6,190,3650,1,2007


# revisar el dataset

In [5]:
print(df_csv.shape)
print(df_csv.columns.tolist())
print(df_csv.dtypes)

(333, 9)
['id', 'species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
id                     int64
species                int64
island                 int64
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm      int64
body_mass_g            int64
sex                    int64
year                   int64
dtype: object


# penguins_processed

In [6]:
df_processed = df_csv.copy()

# Quitar nulos por seguridad
df_processed = df_processed.dropna().copy()

# Quitar id porque no aporta al modelo
df_processed = df_processed.drop(columns=["id"])

print(df_processed.shape)
print(df_processed.head())
print(df_processed.dtypes)

(333, 8)
   species  island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0        1       1            39.1           18.7                181   
1        1       1            39.5           17.4                186   
2        1       1            40.3           18.0                195   
3        1       1            36.7           19.3                193   
4        1       1            39.3           20.6                190   

   body_mass_g  sex  year  
0         3750    1  2007  
1         3800    0  2007  
2         3250    0  2007  
3         3450    0  2007  
4         3650    1  2007  
species                int64
island                 int64
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm      int64
body_mass_g            int64
sex                    int64
year                   int64
dtype: object


# guardar el raw en la base de datos

In [7]:
df_processed.to_sql("penguins_processed", engine, if_exists="replace", index=False)
print("penguins_processed cargada correctamente")

penguins_processed cargada correctamente


# validar datos procesados

In [8]:
df = pd.read_sql("SELECT * FROM penguins_processed", engine)

print(df.shape)
print(df.head())
print(df.dtypes)

(333, 8)
   species  island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0        1       1            39.1           18.7                181   
1        1       1            39.5           17.4                186   
2        1       1            40.3           18.0                195   
3        1       1            36.7           19.3                193   
4        1       1            39.3           20.6                190   

   body_mass_g  sex  year  
0         3750    1  2007  
1         3800    0  2007  
2         3250    0  2007  
3         3450    0  2007  
4         3650    1  2007  
species                int64
island                 int64
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm      int64
body_mass_g            int64
sex                    int64
year                   int64
dtype: object


# Split

In [9]:
X = df.drop(columns=["species"])
y = df["species"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())

##Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

X shape: (333, 7)
y shape: (333,)
Features: ['island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex', 'year']
Train: (266, 7) (266,)
Test : (67, 7) (67,)


# definir experimentos

In [10]:
experiments = []

# SVM
for C in [0.1, 1, 10]:
    for kernel in ["linear", "rbf"]:
        for gamma in ["scale", "auto"]:
            experiments.append({
                "model_name": "SVM",
                "params": {
                    "C": C,
                    "kernel": kernel,
                    "gamma": gamma
                }
            })

# RandomForest
for n_estimators in [50, 100, 200]:
    for max_depth in [3, 5, 10, None]:
        experiments.append({
            "model_name": "RandomForest",
            "params": {
                "n_estimators": n_estimators,
                "max_depth": max_depth,
                "random_state": 42
            }
        })

# LogisticRegression
for C in [0.1, 1, 10]:
    for solver in ["lbfgs", "newton-cg", "saga"]:
        experiments.append({
            "model_name": "LogisticRegression",
            "params": {
                "C": C,
                "solver": solver,
                "max_iter": 1000,
                "random_state": 42
            }
        })

print("Total corridas:", len(experiments))

Total corridas: 33


# entrenar y registrar en MLflow

In [11]:
results = []
best_f1 = -1
best_model = None
best_model_name = None
best_run_id = None

for i, exp in enumerate(experiments, start=1):
    model_name = exp["model_name"]
    params = exp["params"]

    with mlflow.start_run(run_name=f"{model_name}_run_{i}") as run:

        if model_name == "SVM":
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", SVC(**params))
            ])

        elif model_name == "RandomForest":
            model = RandomForestClassifier(**params)

        elif model_name == "LogisticRegression":
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(**params))
            ])

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall = recall_score(y_test, y_pred, average="weighted")
        f1 = f1_score(y_test, y_pred, average="weighted")

        mlflow.log_param("model_name", model_name)
        for k, v in params.items():
            mlflow.log_param(k, v)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model"
        )

        results.append({
            "run_id": run.info.run_id,
            "model_name": model_name,
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "params": params
        })

        if f1 > best_f1:
            best_f1 = f1
            best_model = model
            best_model_name = model_name
            best_run_id = run.info.run_id

print("Entrenamiento terminado")
print("Mejor modelo:", best_model_name)
print("Mejor f1:", best_f1)
print("Best run_id:", best_run_id)

2026/03/23 01:51:43 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/03/23 01:51:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instea

🏃 View run SVM_run_1 at: http://mlflow:5000/#/experiments/1/runs/f980f3da37c44322baeca78934eb6408
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/23 01:51:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 01:51:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run SVM_run_2 at: http://mlflow:5000/#/experiments/1/runs/558e2396bea14cbab50826288c98944c
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/23 01:52:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 01:52:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run LogisticRegression_run_31 at: http://mlflow:5000/#/experiments/1/runs/daea0b26ca1b41c8ba175f73105db389
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/23 01:55:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 01:55:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression_run_32 at: http://mlflow:5000/#/experiments/1/runs/7f2772fb45f244ddb5dc4cd7004b781f
🧪 View experiment at: http://mlflow:5000/#/experiments/1


2026/03/23 01:55:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/23 01:55:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LogisticRegression_run_33 at: http://mlflow:5000/#/experiments/1/runs/f6c7ae5bb62d404ea40c6bcdb602f9ba
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Entrenamiento terminado
Mejor modelo: SVM
Mejor f1: 1.0
Best run_id: f980f3da37c44322baeca78934eb6408


# ver resultados ordenados

In [12]:
results_df = pd.DataFrame(results).sort_values(by="f1_score", ascending=False)
results_df.head(10)

,run_id,model_name,accuracy,precision,recall,f1_score,params
0,f980f3da37c44322baeca78934eb6408,SVM,1.0,1.0,1.0,1.0,"{'C': 0.1, 'kernel': 'linear', 'gamma': 'scale'}"
1,558e2396bea14cbab50826288c98944c,SVM,1.0,1.0,1.0,1.0,"{'C': 0.1, 'kernel': 'linear', 'gamma': 'auto'}"
4,b9a29b80fb034c7daa9b04e032d18430,SVM,1.0,1.0,1.0,1.0,"{'C': 1, 'kernel': 'linear', 'gamma': 'scale'}"
6,d34157b29b9a46e088ff2b4cd6a189f4,SVM,1.0,1.0,1.0,1.0,"{'C': 1, 'kernel': 'rbf', 'gamma': 'scale'}"
5,57eebba5b6db4e03a40d90273260b56e,SVM,1.0,1.0,1.0,1.0,"{'C': 1, 'kernel': 'linear', 'gamma': 'auto'}"
7,19a1a35dda92428194f63fb06d1fb442,SVM,1.0,1.0,1.0,1.0,"{'C': 1, 'kernel': 'rbf', 'gamma': 'auto'}"
14,5255b3ae7ea241bfa330b1388b4fe53a,RandomForest,1.0,1.0,1.0,1.0,"{'n_estimators': 50, 'max_depth': 10, 'random_..."
11,0e682cf5e3be4db0a2c90c56453a88f7,SVM,1.0,1.0,1.0,1.0,"{'C': 10, 'kernel': 'rbf', 'gamma': 'auto'}"
10,769525df0e99468b9ce1f2ee62eebdbc,SVM,1.0,1.0,1.0,1.0,"{'C': 10, 'kernel': 'rbf', 'gamma': 'scale'}"
15,61feea2eec2c460eb868a3e6a86262ec,RandomForest,1.0,1.0,1.0,1.0,"{'n_estimators': 50, 'max_depth': None, 'rando..."


# registrar el mejor modelo en MLflow Registry

In [13]:
model_uri = f"runs:/{best_run_id}/model"
registered_model_name = "penguins-best-model"

mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name
)

print("Modelo registrado en MLflow Registry")
print("Nombre:", registered_model_name)
print("URI:", model_uri)

Successfully registered model 'penguins-best-model'.
2026/03/23 01:55:53 WARNING mlflow.tracking._model_registry.fluent: Run with id f980f3da37c44322baeca78934eb6408 has no artifacts at artifact path 'model', registering model based on models:/m-4404a8365cd04c84a52ab080ce31a20a instead
2026/03/23 01:55:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: penguins-best-model, version 1
Created version '1' of model 'penguins-best-model'.


Modelo registrado en MLflow Registry
Nombre: penguins-best-model
URI: runs:/f980f3da37c44322baeca78934eb6408/model


# guardar resultados en CSV

In [14]:
results_df.to_csv("/app/notebooks/penguins_experiment_results.csv", index=False)
print("Resultados exportados a CSV")

Resultados exportados a CSV
